In [77]:
import tensorflow as tf
from tensorflow.keras import layers, models, metrics
import numpy as np

In [78]:
# Dummy Data Tabular [jam_tidur, aktivitas_fisik, jam_kerja]
X_tabular = np.array([
    [6.0, 1.0, 8.0],
    [3.0, 0.0, 12.0],
    [8.0, 2.0, 5.0]
])

In [79]:
# Dummy data teks catatan harian
X_text = np.array([
    "Hari ini capek banget, tapi semua kerjaan beres sih",
    "Mati rasa, tugas banyak yang belum selesai, dosen galak, pengen nyerah deh",
    "Santai banget hari ini, asik juga bisa mabar dan nongki"
], dtype=object)

In [80]:
# LAKUKAN PROSES DATA PREPROCESSING TEKS LAINNYA

In [81]:
# Label Target 0 (stress), 1 (normal), 2 (happy)
y = np.array([1, 0, 2])

# Lakukan one-hot encoding
y_one_hot = tf.keras.utils.to_categorical(y, num_classes=3)

In [82]:
# Layer untuk proses ekstraksi fitur secara otomatis
vectorize_layer = layers.TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=20
)

# Adaptasi X_test dengan vocab
vectorize_layer.adapt(X_text)

In [83]:
# Contoh arsitektur multi-input dengan functional API tensorflow

# Cabang 1 jalur numerik (data tabular)
input_tabular = layers.Input(shape=(3,), name="input_tabular")
x1 = layers.Dense(16, activation='relu')(input_tabular)
x1 = layers.Dense(8, activation='relu')(x1)

# Cabang 2 jalur teks
input_text = layers.Input(shape=(1,), dtype=tf.string, name="input_text")
x2 = vectorize_layer(input_text)
x2 = layers.Embedding(input_dim=1000, output_dim=16)(x2)

# Menggunakan LSTM agar model mengerti urutan dan konteks kalimat
x2 = layers.LSTM(16)(x2)
x2 = layers.Dense(8, activation='relu')(x2)

# Titik temu kedua cabang (penggabungan atau concatenation)
merged = layers.concatenate([x1, x2])

# Layer akhir untuk klasifikasi
z = layers.Dense(16, activation='relu')(merged)
z = layers.Dropout(0.2)(z)
output = layers.Dense(3, activation='softmax', name="output_model")(z)

In [84]:
# Proses compile dan training
model = models.Model(inputs=[input_tabular, input_text], outputs=output)

model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=['accuracy', 
                       metrics.F1Score(average="weighted", name="f1_score")
                       ]
             )

In [85]:
model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_text          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 20)        │          0 │ input_text[0][0]  │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_tabular       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 20, 16)    │     16,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, 16)        │         64 │ input_tabular[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ (None, 16)        │      2,112 │ embedding_8[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_32 (Dense)    │ (None, 8)         │        136 │ dense_31[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_33 (Dense)    │ (None, 8)         │        136 │ lstm_8[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, 16)        │          0 │ dense_32[0][0],   │
│ (Concatenate)       │                   │            │ dense_33[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_34 (Dense)    │ (None, 16)        │        272 │ concatenate_8[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 16)        │          0 │ dense_34[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_model        │ (None, 3)         │         51 │ dropout_7[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 18,771 (73.32 KB)

 Trainable params: 18,771 (73.32 KB)

 Non-trainable params: 0 (0.00 B)

In [86]:
history = model.fit(
    {"input_tabular": X_tabular, "input_text": X_text},
    y_one_hot,
    epochs=10,
    verbose=1
)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 1.0516
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 1.0732
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 1.0311
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 1.0451
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 1.0283
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 1.0156
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 0.9860
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3333 - f1_score: 0.1667 - loss: 0.9674
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.6667 - f1_score: 0.5556 - loss: 0.9432
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.3333 - f1_score: 0.1